# 04c — 第四讲基础习题：高斯过程回归与贝叶斯优化巩固练习

本套习题帮助你一步步巩固第四讲（`04a`–`04b`）中高斯过程回归（GPR）、核函数、预测不确定性量化以及贝叶斯优化采集函数（UCB / EI）的核心操作。每道题只需填写 **2–5 行代码**，并附有预期数值提示供你自查。完成后可对照 `solutions/04_exercises_solutions.ipynb` 查看完整解答。

---

## 0. 公共环境准备

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel, ConstantKernel as C

plt.rcParams["figure.dpi"] = 110

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（本地与 Colab 通用）
import pathlib, sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 第一部分：核函数与高斯过程回归基础（对应 `04a`）

### 🏋️ 习题 1：计算一维点集的 RBF 核协方差矩阵

给定三个退火温度点 `X_pts = np.array([[1.0], [2.0], [5.0]])`（单位：归一化温度），根据 RBF 核公式：
$$k(x_i, x_j) = \exp\left(-\frac{(x_i - x_j)^2}{2\ell^2}\right)$$
请使用 `RBF(length_scale=1.0)`（可直接调用 `kernel(X_pts)`）计算这 3 个点之间的 $3 \times 3$ 协方差矩阵 `K_mat` 并打印（保留 3 位小数）。
- **预期输出**：对角线全为 `1.000`；`x=1.0` 与 `x=2.0` 的相关系数为 `0.607`；`x=1.0` 与 `x=5.0` 的相关系数为 `0.000`。

In [ ]:
X_pts = np.array([[1.0], [2.0], [5.0]])

# 在下方填写你的代码：


### 🏋️ 习题 2：使用 `GaussianProcessRegressor` 输出预测均值与标准差

下方给出了某钙钛矿薄膜在 5 个旋涂转速 `X_obs` 下测得的发光量子产率 `y_obs`。
1. 实例化 `GaussianProcessRegressor(kernel=C(1.0) * RBF(1.0), alpha=1e-4, random_state=0)` 并在 `(X_obs, y_obs)` 上拟合；
2. 调用 `gp.predict(X_test_pts, return_std=True)` 预测两个待测点 `X_test_pts = np.array([[2.1], [6.0]])` 的均值 `mu` 和标准差 `std` 并打印。
- **思考**：为什么 `x = 2.1`（紧挨着已测点 `2.0`）的预测标准差极小（约 `0.04`），而 `x = 6.0`（处在空白区中央）的预测标准差很大（约 `0.56`）？

In [ ]:
X_obs = np.array([[1.0], [2.0], [3.0], [8.5], [9.5]])
y_obs = np.array([0.45, 0.78, 0.62, 0.30, 0.25])
X_test_pts = np.array([[2.1], [6.0]])

# 在下方填写你的代码：


### 🏋️ 习题 3：绘制 GPR 的预测均值曲线与 95% 置信区间

延续上一题训练好的 `gp` 模型：
1. 在密集网格 `X_grid = np.linspace(0, 10, 200).reshape(-1, 1)` 上调用 `gp.predict(X_grid, return_std=True)` 得到 `mu_grid, std_grid`；
2. 画出预测均值曲线 `plt.plot(X_grid, mu_grid)`，用 `plt.fill_between(X_grid.ravel(), mu_grid - 1.96 * std_grid, mu_grid + 1.96 * std_grid, alpha=0.2)` 画出 95% 置信区间，并用红点标出已测点 `(X_obs, y_obs)`。

In [ ]:
X_grid = np.linspace(0, 10, 200).reshape(-1, 1)

# 在下方填写你的绘图代码：


### 🏋️ 习题 4：用 `WhiteKernel` 从带噪实验数据中自动估计测量误差

下方生成了 20 个带有真实噪声标准差 `0.20`（即方差 `noise_level = 0.04`）的实验观测点 `(X_noisy, y_noisy)`。
请使用复合核 `kernel = C(1.0) * RBF(1.0) + WhiteKernel(noise_level=0.05)` 拟合 `GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=5, random_state=0)`，并打印拟合后的核参数 `gp_noisy.kernel_`。
- **预期输出**：`WhiteKernel` 学到的 `noise_level` 约为 `0.036`（对应的标准差 $\sqrt{0.036} \approx 0.19$，与真实噪声 `0.20` 非常吻合）。

In [ ]:
rng = np.random.RandomState(42)
X_noisy = np.linspace(0, 8, 20).reshape(-1, 1)
y_noisy = np.sin(X_noisy).ravel() + rng.normal(0, 0.20, size=20)

# 在下方填写你的代码：


---

## 第二部分：贝叶斯优化采集函数与主动推荐（对应 `04b`）

### 🏋️ 习题 5：手写 UCB 采集函数并在 3 个候选方案中做出选择

假设我们已经用 GPR 对三个未测候选配方的产率进行了预测，得到的预测均值与标准差分别为：
- **配方 A（稳健高均值）**：`mu = 82.0%`, `std = 2.0%`
- **配方 B（中等均值、中等不确定性）**：`mu = 78.0%`, `std = 6.0%`
- **配方 C（全新结构盲区、极高不确定性）**：`mu = 72.0%`, `std = 12.0%`

请根据公式 $\text{UCB} = \mu + \kappa \cdot \sigma$，分别计算当：1) `kappa = 0.5`（偏向利用）和 2) `kappa = 2.0`（偏向探索）时三个配方的 UCB 得分，看看各自会推荐哪个配方。

In [ ]:
names = ["配方 A", "配方 B", "配方 C"]
mu  = np.array([82.0, 78.0, 72.0])
std = np.array([2.0,  6.0,  12.0])

# 在下方填写你的代码：


### 🏋️ 习题 6：计算期望提升（Expected Improvement, EI）并推荐下一个温度点

延续习题 2 和习题 3 中的 `gp` 模型（当前已测 5 个点的最高量子产率为 `y_best = y_obs.max() = 0.78`）：
1. 使用下方已定义好的 `acq_ei(mu_grid, std_grid, y_best)` 计算网格 `X_grid` 上每个点的 EI 值；
2. 找出使 EI 最大的网格点坐标 `next_x = X_grid[np.argmax(ei_vals), 0]`，对比它与纯利用（Greedy，`X_grid[np.argmax(mu_grid), 0]`）推荐的位置。

In [ ]:
def acq_ei(mu, std, y_best, xi=0.01):
    with np.errstate(divide="ignore", invalid="ignore"):
        imp = mu - y_best - xi
        Z = np.where(std > 1e-9, imp / std, 0.0)
        return np.where(std > 1e-9, imp * norm.cdf(Z) + std * norm.pdf(Z), 0.0)

# 在下方填写你的代码：


### 🏋️ 习题 7：计算分子指纹的 Tanimoto 相似度核矩阵

下方给出了三个简单分子的 8 位示意二值指纹矩阵 `F`（3 行 8 列）：
- 第 0 行：`[1, 1, 0, 0, 1, 0, 0, 0]`
- 第 1 行：`[1, 1, 1, 0, 1, 0, 0, 0]`
- 第 2 行：`[0, 0, 0, 1, 0, 1, 1, 1]`

请根据向量化公式 $K(A, B) = \frac{A B^T}{\|A\|^2 + \|B\|^2 - A B^T}$ 计算 $3 \times 3$ 的 Tanimoto 核矩阵并打印。
- **预期输出**：分子 0 与分子 1 的相似度为 `0.75`，分子 0 与分子 2 的相似度为 `0.00`。

In [ ]:
F = np.array([
    [1., 1., 0., 0., 1., 0., 0., 0.],
    [1., 1., 1., 0., 1., 0., 0., 0.],
    [0., 0., 0., 1., 0., 1., 1., 1.],
])

# 在下方填写你的代码：


### 🏋️ 习题 8：在 Buchwald–Hartwig 偶联反应子集上执行 10 步贝叶斯优化

下方代码从 `data/bh-reactions.csv` 中取出了前 300 个反应構成候选池 `(X_sub, y_sub)`（最高产率为 `99.9%`），并指定了 3 个初始反应 `obs = [0, 10, 20]`（初始最高产率仅 `45.7%`）。
请补全 `for step in range(10):` 循环内的 4 行核心代码：在 `X_sub[obs], y_sub[obs]` 上拟合 `gp`，对未测反应 `X_sub[unseen]` 预测 `mu, std`，用 `acq_ucb(mu, std, kappa=2.0)` 选出最高分反应加入 `obs`。
- **预期输出**：经过 10 步主动推荐后，找到的最高反应产率从 `45.7%` 跃升至 **`98.0%`** 以上！

In [ ]:
for p in ["data/bh-reactions.csv", "../data/bh-reactions.csv", "bh-reactions.csv"]:
    if os.path.exists(p):
        bh = pd.read_csv(p)
        break

X_all = pd.get_dummies(bh[["ligand", "additive", "base", "aryl halide"]], dtype=float).values
X_sub = X_all[:300]
y_sub = bh["yield"].values[:300]

obs = [0, 10, 20]
print(f"初始 3 个反应的最高产率: {y_sub[obs].max():.1f}% (子集全局最高: {y_sub.max():.1f}%)")

for step in range(1, 11):
    unseen = [i for i in range(len(y_sub)) if i not in obs]
    gp = GaussianProcessRegressor(kernel=C(1.0) * RBF(2.0) + WhiteKernel(1.0), normalize_y=True, random_state=0)
    # 在下方补全拟合、预测与选点代码（约 3~4 行）：
    
print(f"10 步 BO 后找到的最高产率: {y_sub[obs].max():.1f}%")